In [1]:
# =========================================
# 01 — Data Prep (Olist) | Colab
# Mount Drive, load CSVs, build orders_fact, export curated tables
# =========================================

import os
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- Paths (adjust only if needed) ---
DATA_DIR = "/content/drive/MyDrive/Analytics_Portfolio/Olist Case/data"
EXPORT_DIR = "/content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports"

os.makedirs(EXPORT_DIR, exist_ok=True)

In [3]:
# --- Expected files ---
FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv"
}

print("DATA_DIR:", DATA_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("\nChecking files...")

missing = []
for k, f in FILES.items():
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} (missing)")
        missing.append(f)

DATA_DIR: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/data
EXPORT_DIR: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports

Checking files...
✅ olist_orders_dataset.csv
✅ olist_order_items_dataset.csv
✅ olist_customers_dataset.csv
✅ olist_products_dataset.csv


In [4]:
# --- Show what's inside the data folder (helps debugging) ---
print("\nFiles found in data folder:")
for f in sorted(os.listdir(DATA_DIR)):
    print(" -", f)


Files found in data folder:
 - olist_customers_dataset.csv
 - olist_order_items_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv


In [5]:
# --- Required files) ---
REQUIRED = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
}

In [6]:
# --- Validate presence ---
missing = [fname for fname in REQUIRED.values() if not os.path.exists(os.path.join(DATA_DIR, fname))]
if missing:
    raise FileNotFoundError(f"Missing required file(s) in DATA_DIR: {missing}")

print("\n✅ All required files found.")


✅ All required files found.


In [7]:
# --- Load data ---
orders = pd.read_csv(os.path.join(DATA_DIR, REQUIRED["orders"]))
items = pd.read_csv(os.path.join(DATA_DIR, REQUIRED["items"]))
customers = pd.read_csv(os.path.join(DATA_DIR, REQUIRED["customers"]))
products = pd.read_csv(os.path.join(DATA_DIR, REQUIRED["products"]))

print("\nShapes:")
print("orders:", orders.shape)
print("items:", items.shape)
print("customers:", customers.shape)
print("products:", products.shape)


Shapes:
orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)


In [8]:
# --- Parse order date columns ---
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

In [9]:
# --- Keep delivered orders (standard for retention) ---
delivered = orders.loc[orders["order_status"] == "delivered"].copy()

print("\nOrder status counts:")
print(orders["order_status"].value_counts())
print("\nDelivered orders:", delivered.shape)


Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Delivered orders: (96478, 8)


In [10]:
# --- Revenue per order from items ---
items["item_revenue"] = items["price"].fillna(0) + items["freight_value"].fillna(0)

order_revenue = (
    items.groupby("order_id", as_index=False)
        .agg(
            items_count=("order_item_id", "count"),
            revenue=("item_revenue", "sum"),
            price_sum=("price", "sum"),
            freight_sum=("freight_value", "sum"),
        )
)

In [11]:
# --- orders_fact (order grain) ---
orders_fact = delivered.merge(order_revenue, on="order_id", how="left")

In [12]:
# Add customer_unique_id
orders_fact = orders_fact.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

In [13]:
# Time fields
orders_fact["order_month"] = orders_fact["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders_fact["order_date"] = orders_fact["order_purchase_timestamp"].dt.date

In [14]:
# Fill null revenue (rare mismatches)
orders_fact["revenue"] = orders_fact["revenue"].fillna(0)

In [15]:
# --- Export (for Power BI + next notebooks) ---
orders_fact_path = os.path.join(EXPORT_DIR, "orders_fact.csv")
orders_fact.to_csv(orders_fact_path, index=False)
print("\n✅ Exported:", orders_fact_path)

orders_fact_light = orders_fact[
    ["order_id", "customer_unique_id", "order_purchase_timestamp", "order_month", "revenue", "items_count"]
].copy()

orders_fact_light_path = os.path.join(EXPORT_DIR, "orders_fact_light.csv")
orders_fact_light.to_csv(orders_fact_light_path, index=False)
print("✅ Exported:", orders_fact_light_path)

orders_fact.head()


✅ Exported: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports/orders_fact.csv
✅ Exported: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports/orders_fact_light.csv


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,items_count,revenue,price_sum,freight_sum,customer_unique_id,order_month,order_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,38.71,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-10-02
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,141.46,118.70,22.76,af07308b275d755c9edb36a90c618231,2018-07,2018-07-24
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,179.12,159.90,19.22,3a653a41f6f9fc3d2a113cf8398680e8,2018-08,2018-08-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,72.20,45.00,27.20,7c142cf63193a1473d2e66489a9ae977,2017-11,2017-11-18
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,28.62,19.90,8.72,72632f0f9dd73dfee390c9b22eb56dd6,2018-02,2018-02-13
